DSCI 552 Homework 2

Name: Brynn Dafoe GitHub Username: brynndafoe02 USD ID: 3109-6692-10

In [61]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score, mean_absolute_error
from itertools import combinations
import statsmodels.api as sm

ModuleNotFoundError: No module named 'statsmodels'

In [ ]:
df = pd.read_excel("../data/CCPP/Folds5x2_pp.xlsx", header=0)
#df.head()

Question 1.b.i
- How many rows are in this data set? How many columns? What do the rows and columns represent?

There are 9568 rows and 5 columns. The first 4 columns represent the features: Temperture (T or AT), Exhaust Vaccuum (V), Ambient Pressure (AP), and Relative Humidity (RH). The last column represents the net hourly electrical energy output (EP or PE), which is predicted by the first 4 variables. Each row represents one hourly measurement. 

Question 1.b.ii
- Make pairwise scatterplots of all the variables in the data set including the predictors (IVs) with the DV
- Describe your findings

In [ ]:
sns.pairplot(df)
plt.show()

For seaborn.pairplot, the diagonal shows the distribution of each variable alone. The AT the distribution is almost normal, but it is slightly split in the middle with a good spread. V is more split down the middle with some possible outliers / possibly abnormally elevated values. AP has a good basically normal distribution. RH's distribution is skewed to the right towards the higher values, but still in somewhat of a bell shape. PE is distributed similar to AT, but slightly skewed to the left towards the lower values.

The rest of the plots show the relationships. The last colomn shows the relationship between each IV and the DV. 

AT x PE: The relationship seems to be negative, as AT values rise EP values lower. It also has a decently tight pattern showing a stronger relationship, and also it seems to be pretty linear as well since it is pretty straight.

V x PE: The relationship also seems to be negative. This relationship is less linear as it shows a bit of a curved slope downwards. It also shows a couple oddities like the line of values in a straight line where V stays the same where PE increases. There is also a small grouping along the bottom that could be outliers.

AP x PE: This relationship is clearly weaker, but it seems to be a positive relationship. It is very slightly curved although it may be too weak to say for sure. 

RH x PE: This relationship is the weakest. I don't think I can say for sure whether it is positive or negative.

Question 1.b.iii
- What are the mean, median, range, first quartile, third quartile, and interquartile range of each of the variables in the dataset?
- Summarize them in a table

In [ ]:
summary_stats = df.describe()

AT_mean = summary_stats.loc["mean", "AT"]
AT_median = summary_stats.loc["50%", "AT"]
AT_range = summary_stats.loc["max", "AT"] - summary_stats.loc["min", "AT"]
AT_1Q = summary_stats.loc["25%", "AT"]
AT_3Q = summary_stats.loc["75%", "AT"]
AT_IQR = AT_3Q - AT_1Q

V_mean = summary_stats.loc["mean", "V"]
V_median = summary_stats.loc["50%", "V"]
V_range = summary_stats.loc["max", "V"] - summary_stats.loc["min", "V"]
V_1Q = summary_stats.loc["25%", "V"]
V_3Q = summary_stats.loc["75%", "V"]
V_IQR = V_3Q - V_1Q

AP_mean = summary_stats.loc["mean", "AP"]
AP_median = summary_stats.loc["50%", "AP"]
AP_range = summary_stats.loc["max", "AP"] - summary_stats.loc["min", "AP"]
AP_1Q = summary_stats.loc["25%", "AP"]
AP_3Q = summary_stats.loc["75%", "AP"]
AP_IQR = AP_3Q - AP_1Q

RH_mean = summary_stats.loc["mean", "RH"]
RH_median = summary_stats.loc["50%", "RH"]
RH_range = summary_stats.loc["max", "RH"] - summary_stats.loc["min", "RH"]
RH_1Q = summary_stats.loc["25%", "RH"]
RH_3Q = summary_stats.loc["75%", "RH"]
RH_IQR = RH_3Q - RH_1Q

PE_mean = summary_stats.loc["mean", "PE"]
PE_median = summary_stats.loc["50%", "PE"]
PE_range = summary_stats.loc["max", "PE"] - summary_stats.loc["min", "PE"]
PE_1Q = summary_stats.loc["25%", "PE"]
PE_3Q = summary_stats.loc["75%", "PE"]
PE_IQR = PE_3Q - PE_1Q

table_stats = pd.DataFrame({"Variable": ["Temperature", "Exhaust Vacuum", "Ambient Pressure", "Relative Humidity", "Electrical Energy Output"], 
                            "Mean": [AT_mean, V_mean, AP_mean, RH_mean, PE_mean], 
                            "Median": [AT_median, V_median, AP_median, RH_median, PE_median], 
                            "Range": [AT_range, V_range, AP_range, RH_range, PE_range], 
                            "1Q": [AT_1Q, V_1Q, AP_1Q, RH_1Q, PE_1Q], 
                            "3Q": [AT_3Q, V_3Q, AP_3Q, RH_3Q, PE_3Q], 
                            "IQR": [AT_IQR, V_IQR, AP_IQR, RH_IQR, PE_IQR]})

print(table_stats)



Question 1.c
- For each predictor, fit a simple linear regression model to predict the response
- Describe your results
- In which of the models is there a statistically significant assocation between the predictor and the response?
- Create some plots to back up your assertions
- Are there any outliers that you would like to remove from your data for each of these regression tasks?

In [ ]:
# X -> predictors (AT, V, AP, RH), array of shape (n_samples, n_features), training
# Y -> to be predicted on (PE / EP), array of shape (n_samples,) or (n_samples, n_targets), targets
# y = mx + b
# score(X, Y, sample weight=None) -> returns coefficient of determination / R^2, will show how well reg line fits data

predictors = ["AT", "V", "AP", "RH"]

for p in predictors:
    X = df[[p]]
    Y = df["PE"]

    lr_model = LinearRegression()
    lr_model.fit(X, Y)

    r2 = lr_model.score(X, Y)

    print(f"Predictor: {p}")
    print(f"Intercept: {lr_model.intercept_}")
    print(f"Slope: {lr_model.coef_[0]}")
    print(f"R^2: {r2}")
    print()

Seeing the results above, it looks like AT has the strongest association with PE with an R^2 of 0.8989 out of 1 with a negative relationship (which is what was suspected from the original plots in 1.b.ii). V has the second strongest relationship with an R^2 of 0.7565 out of 1 and also a negative relationship (also suspected earlier). AP and RH have very weak relationships with PE (also suspected).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 15))
axes = axes.flatten()

for i in range(len(predictors)):
    axes[i].scatter(df[predictors[i]], df["PE"])
    axes[i].set_xlabel(predictors[i])
    axes[i].set_ylabel("PE")

plt.tight_layout()
plt.show()

As can be seen better in these plots, PE plotted against AT and V show stronger / tighter relationships, both with negative relationships. AT clearly has the strongest relationship. As can also be seen, AP and RH show very weak relationships. 

AT, AP, and RH don't really show extreme outliers. There are some points that stray further from the cluster, but I don't think they are far enough away to justify removing them. As for V, there are two areas where V remains constant while PE varies. It forms two straight vertical lines (where V =~ 15 and V =~ 70). The PE x V plot shows more "straighter" variations rather than the rest where the data points look more truly scattered, so if I were to remove any "outliers" I think I would remove these two lines, but I think I could also not justify completely removing them either. 

In summary, I would not remove any outliers, but I am noting the two lines showing up in the PE x V plot. 

Question 1.d
- Fit a multiple regression model to predict the response using all of the predictors
- Describe your results
- For which predictors can we reject the null hypothesis H0 : Bj = 0?

In [ ]:
predictors = ["AT", "V", "AP", "RH"]

X_mult = df[predictors]
Y_mult = df["PE"]

mult_lr_model = LinearRegression()
mult_lr_model.fit(X_mult, Y_mult)

r2_mult = mult_lr_model.score(X_mult, Y_mult)

# just using this conceptually 
# y = B0 + B1x1 + B2x2 + B3x3 + B4x4
# y = int+ slAT + slV  + slAP + slRH
#     int  coef0  coef1  coef2  coef3
print(f"Intercept: {mult_lr_model.intercept_}") # B0
print(f"AT Slope: {mult_lr_model.coef_[0]}") # B1
print(f"V Slope: {mult_lr_model.coef_[1]}") # B2
print(f"AP Slope: {mult_lr_model.coef_[2]}") # B3
print(f"RH Slope: {mult_lr_model.coef_[3]}") # B4
print(f"R^2: {r2_mult}")


So: PE = 454.61 - 1.98(AT) - 0.23(V) + 0.06(AP) - 0.15(RH)

This shows somewhat similar results as the previous two. AT seems to have the strongest relationship with the largest coefficient, and it is again showing as a negative relationship. V and RH have smaller coefficients with a negative relationship and AP has the smallest coefficient with a positive relationship. It seems that AP does not have much pull in this equation due to the small coefficient, which looks correct compared to the other questions (although I may have expected RH to have a smaller magnitude, although that one is still pretty small as well).

The R^2 here is high, showing that these predictors explain about 92% of the variation in PE.